HW 2

In [ ]:
# q1.1
import pandas as pd
import numpy as np
airbnb = pd.read_csv("./airbnb_hw.csv")
# Price Variable
print(airbnb['Price'].head(20))
print(airbnb['Price'].unique()[:30])
# Cleaning
def clean_price(price_value):
    if pd.isna(price_value):
        return np.nan
    price_string = str(price_value)
    price_string = price_string.replace('$', '')
    price_string = price_string.replace(',', '')
    price_string = price_string.strip()
    try:
        return float(price_string)
    except:
        return np.nan
airbnb['Price_clean'] = airbnb['Price'].apply(clean_price)
print(airbnb['Price_clean'].head(20))
missing_count = airbnb['Price_clean'].isna().sum()
print(missing_count)



0     145
1      37
2      28
3     199
4     549
5     149
6     250
7      90
8     270
9     290
10    170
11     59
12     49
13     68
14    285
15     75
16    145
17    100
18    150
19    700
Name: Price, dtype: object
['145' '37' '28' '199' '549' '149' '250' '90' '270' '290' '170' '59' '49'
 '68' '285' '75' '100' '150' '700' '125' '175' '40' '89' '95' '99' '499'
 '120' '79' '110' '180']
0     145.0
1      37.0
2      28.0
3     199.0
4     549.0
5     149.0
6     250.0
7      90.0
8     270.0
9     290.0
10    170.0
11     59.0
12     49.0
13     68.0
14    285.0
15     75.0
16    145.0
17    100.0
18    150.0
19    700.0
Name: Price_clean, dtype: float64
0


In [35]:
#q1.2
police = pd.read_csv('./mn_police_use_of_force.csv')

print(police['subject_injury'].value_counts(dropna=False))  # dropna=False shows NaN count

# Cleaning Yes and No
def clean_injury(value):
    if pd.isna(value):
        return np.nan
    
    value_lower = str(value).lower().strip()
    
    if value_lower in ['yes', 'y', '1', 'true']:
        return 'Yes'
    elif value_lower in ['no', 'n', '0', 'false']:
        return 'No'
    else:
        return np.nan
police['subject_injury_clean'] = police['subject_injury'].apply(clean_injury)
print(police['subject_injury_clean'].value_counts(dropna=False))

missing_count = police['subject_injury_clean'].isna().sum()
total_count = len(police)
proportion_missing = missing_count / total_count
print(proportion_missing)

print(proportion_missing > 0.20)
# this is concerning because more than 20% missing is concerning

# Cross-tabulate with force_type
crosstab = pd.crosstab(
    police['subject_injury_clean'],  # rows
    police['force_type'],             # columns
    margins=True,                     # adds totals
    dropna=False                      # includes NaN as a category
)
print(crosstab)

for force_type in police['force_type'].unique():
    subset = police[police['force_type'] == force_type]
    pct_missing = (subset['subject_injury_clean'].isna().sum() / len(subset)) * 100
print(pct_missing)

subject_injury
NaN    9848
Yes    1631
No     1446
Name: count, dtype: int64
subject_injury_clean
NaN    9848
Yes    1631
No     1446
Name: count, dtype: int64
0.7619342359767892
True
force_type            Baton  Bodily Force  Chemical Irritant  Firearm  \
subject_injury_clean                                                    
No                        0          1093                131        2   
Yes                       2          1286                 41        0   
NaN                       2          7051               1421        0   
All                       4          9430               1593        2   

force_type            Gun Point Display  Improvised Weapon  Less Lethal  \
subject_injury_clean                                                      
No                                   33                 34            0   
Yes                                  44                 40            0   
NaN                                  27                 74           87   
Al

In [44]:
# q1.3
pretrial = pd.read_parquet('./justice_data.parquet')

print(pretrial['WhetherDefendantWasReleasedPretrial'].value_counts(dropna=False))

# Clean into binary 0/1
def clean_release(value):
    if pd.isna(value):
        return np.nan
    
    value_lower = str(value).lower().strip()
    
    if value_lower in ['yes', 'y', '1', '1.0', 'true', 't']:
        return 1
    elif value_lower in ['no', 'n', '0', '0.0', 'false', 'f']:
        return 0
    else:
        return np.nan

pretrial['released_clean'] = pretrial['WhetherDefendantWasReleasedPretrial'].apply(clean_release)

print(pretrial['released_clean'].value_counts(dropna=False))

missing_count = pretrial['released_clean'].isna().sum()
print(f"Missing values: {missing_count} (replaced with np.nan)")
print(f"Percentage missing: {(missing_count/len(pretrial))*100:.2f}%")

WhetherDefendantWasReleasedPretrial
1    19154
0     3801
9       31
Name: count, dtype: int64
released_clean
1.0    19154
0.0     3801
NaN       31
Name: count, dtype: int64
Missing values: 31 (replaced with np.nan)
Percentage missing: 0.13%


In [47]:
# q1.4
# Missing
pretrial['ImposedSentenceAllChargeInContactEvent'].isna().sum()
# Not missing
pretrial['ImposedSentenceAllChargeInContactEvent'].notna().sum()

print(pretrial['SentenceTypeAllChargesAtConvictionInContactEvent'].value_counts(dropna=False))

# Step 3: Understand the relationship
# Key insight: If case was DISMISSED or ACQUITTED, there is NO sentence (not just missing data).
# So missing sentence = 0 for these cases.

# Clean the sentence variable
def clean_sentence(row):
    sentence = row['ImposedSentenceAllChargeInContactEvent']
    sentence_type = row['SentenceTypeAllChargesAtConvictionInContactEvent']
    
    if pd.notna(sentence):
        try:
            return float(sentence)
        except:
            return np.nan
    
    if pd.notna(sentence_type):
        sentence_type_lower = str(sentence_type).lower()
        
        no_conviction_terms = ['dismiss', 'acquit', 'none', 'no sentence', 'no conviction']
        
        if any(term in sentence_type_lower for term in no_conviction_terms):
            return 0

    return np.nan

pretrial['sentence_clean'] = pretrial.apply(clean_sentence, axis=1)

# Step 6: Compare before and after
missing_before = pretrial['ImposedSentenceAllChargeInContactEvent'].isna().sum()
missing_after = pretrial['sentence_clean'].isna().sum()
converted_to_zero = missing_before - missing_after
print(converted_to_zero)



SentenceTypeAllChargesAtConvictionInContactEvent
4    8779
0    8720
1    4299
2     914
9     274
Name: count, dtype: int64
-9053


In [ ]:
#q2
import matplotlib.pyplot as plt

sharks = pd.read_excel('./GSAF5.xls', engine='xlrd')
sharks = sharks.dropna(axis=1, how='all')
sharks['Year_clean'] = sharks['Year'].apply(lambda x: int(''.join(c for c in str(x) if c.isdigit())) if pd.notna(x) else np.nan)
sharks = sharks[sharks['Year_clean'] >= 1940]
early_avg = len(sharks[sharks['Year_clean'] < 1980]) / 40
recent_avg = len(sharks[sharks['Year_clean'] >= 1980]) / 40

def clean_age(age):
    if pd.isna(age):
        return np.nan
    age_str = ''.join(c for c in str(age).split('-')[0] if c.isdigit())
    if age_str == '':
        return np.nan
    try:
        return int(age_str)
    except:
        return np.nan

sharks['Age_clean'] = sharks['Age'].apply(clean_age)
plt.hist(sharks['Age_clean'].dropna(), bins=30)
plt.savefig('age_histogram.png')
plt.close()

sharks['Sex_clean'] = sharks['Sex'].apply(lambda x: 'M' if str(x).upper() in ['M','MALE'] else 'F' if str(x).upper() in ['F','FEMALE'] else 'Unknown')
prop_male = (sharks['Sex_clean'] == 'M').sum() / sharks[sharks['Sex_clean'] != 'Unknown'].shape[0]
print(f"Proportion male: {prop_male:.4f}")


sharks['Type_clean'] = sharks['Type'].apply(lambda x: 'Unprovoked' if 'unprovoked' in str(x).lower() else 'Provoked' if 'provoked' in str(x).lower() else 'Unknown')
prop_unprovoked = (sharks['Type_clean'] == 'Unprovoked').sum() / len(sharks)
print(f"Proportion unprovoked: {prop_unprovoked:.4f}")


sharks['Fatal_clean'] = sharks['Fatal Y/N'].apply(lambda x: 'Y' if str(x).upper() in ['Y','YES'] else 'N' if str(x).upper() in ['N','NO'] else 'Unknown')

print(sharks.columns.tolist())

print(sharks[sharks['Type_clean']=='Unprovoked']['Sex_clean'].value_counts()) 
print(pd.crosstab(sharks['Type_clean'], sharks['Fatal_clean'], normalize='index')*100) 
print(pd.crosstab(sharks['Sex_clean'], sharks['Fatal_clean'], normalize='index')*100) 

sharks['is_white'] = sharks['Species '].apply(lambda x: 'white' in str(x).lower() if pd.notna(x) else False)
prop_white = sharks['is_white'].sum() / len(sharks)
print(f"Proportion white: {prop_white:.4f}")


Proportion male: 0.8749
Proportion unprovoked: 0.7365
['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity', 'Name', 'Sex', 'Age', 'Injury', 'Fatal Y/N', 'Time', 'Species ', 'Source', 'pdf', 'href formula', 'href', 'Case Number', 'Case Number.1', 'original order', 'Unnamed: 21', 'Unnamed: 22', 'Year_clean', 'Age_clean', 'Sex_clean', 'Type_clean', 'Fatal_clean']
Sex_clean
M          4267
F           672
Unknown     170
Name: count, dtype: int64
Fatal_clean          N    Unknown          Y
Type_clean                                  
Provoked     95.133438   1.726845   3.139717
Unknown      37.363560  46.431570  16.204870
Unprovoked   74.633001   1.624584  23.742415
Fatal_clean          N    Unknown          Y
Sex_clean                                   
F            75.849057   8.805031  15.345912
M            70.120525   8.328836  21.550639
Unknown      62.264151  19.554031  18.181818
Proportion white: 0.1087


q3
1. The paper talks about data tidying, main idea is structuring datasets to facilitate analysis by organizing them so each variable is a column, each observation is a row, and each type of observational unit is a table.

2. The tidy data standard provides a consistent way to organize data values within a dataset, making initial data cleaning easier and simplifying the development of data analysis tools that work well together without requiring constant translation between different formats.

3. "Like families, tidy datasets are all alike but every messy dataset is messy in its own way" means that tidy data follows consistent rules, but messy data can be disorganized in countless different ways. The second sentence means that while it's usually obvious what counts as an observation or variable in a specific dataset, creating a universal definition that works for all datasets is surprisingly challenging because the same data can be conceptualized differently depending on context.

4. A value is a single number or string, a variable contains all values measuring the same attribute across units (like height or temperature), and an observation contains all values measured on the same unit across different attributes (like all measurements for one person).

5. Tidy data is defined by three criteria: (1) each variable forms a column, (2) each observation forms a row, and (3) each type of observational unit forms a table.

6. The 5 problems are: (1) column headers are values not variable names, (2) multiple variables in one column, (3) variables in both rows and columns, (4) multiple observational types in one table, (5) one observational type in multiple tables. Table 4 is messy because income levels are stored as column headers rather than as a single income variable. Melting turns columns into rows by converting column headers into a new variable and stacking the data values into another variable.

7. Table 11 is messy because it has variables stored in both columns and rows, with day numbers as column headers instead of values in a variable. Table 12 is tidy because after melting and casting, each variable is in its own column and each row represents one day's observations.

8. The chicken-and-egg problem is that tidy data is only useful with tidy tools, and tidy tools only work with tidy data, making it hard to change either independently and potentially trapping us in a local maximum. Wickham hopes future work will use methodologies from human factors and user-centered design fields to better understand the cognitive aspects of data analysis and develop even better data storage strategies and tools.